# Machine Learning Models
## Public Compliance Data Analysis - MBA Thesis

**Objective:** Build predictive models for compliance analysis:
- Regression models (predict sanctions rate)
- Classification models (high/low compliance risk)
- Feature importance analysis
- Model evaluation and comparison

In [ ]:
# --- AUTO-GENERATED DEPENDENCY INSTALL ---
# Installs all project dependencies on first run (Colab, fresh environments, etc).
# Idempotent: pip skips anything already installed.
# To regenerate this cell, run: python scripts/inject_pip_install.py

import subprocess
import sys
from pathlib import Path

_req = Path.cwd().parent / "requirements.txt"
if not _req.exists():
    _req = Path.cwd() / "requirements.txt"

if _req.exists():
    print(f"Installing dependencies from {_req.name if _req.exists() else "requirements.txt"} ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(_req)])
    print("Dependencies ready.")
else:
    print("requirements.txt not found. Install manually: pip install -r requirements.txt")


## Step-by-step (Aula style)

1. Packages and environment setup
2. Reproducibility
3. Data loading
4. Analysis blocks
5. Summary and interpretation


# Packages


In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.model_selection import GridSearchCV

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

# Try local loader first, fallback to S3 if available
from src.analysis.local_data_loader import LocalGoldDataLoader as GoldDataLoader

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
%matplotlib inline

pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 4)
pd.set_option('display.float_format', '{:.4f}'.format)


In [ ]:
import matplotlib as mpl
mpl.rcParams['axes.formatter.useoffset'] = False
mpl.rcParams['axes.formatter.limits'] = (-99, 99)


# Reproducibility


In [ ]:
import os
import random

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

print(f"Reproducibility seed fixed at {SEED}")


In [ ]:
import json as _json

_rtcfg_path = os.path.join('..', 'config', 'runtime_config.json')
if os.path.exists(_rtcfg_path):
    with open(_rtcfg_path) as _f:
        _rtcfg = _json.load(_f)
else:
    _rtcfg = {}

S3_BUCKET_NAME = os.environ.get('S3_BUCKET_NAME', _rtcfg.get('aws', {}).get('s3_bucket_name', ''))
AWS_PROFILE = os.environ.get('AWS_PROFILE', _rtcfg.get('aws', {}).get('profile', None))


## 1. Load and Prepare Data

In [ ]:
from src.analysis.local_data_loader import LocalGoldDataLoader as GoldDataLoader
loader = GoldDataLoader()

# --- Municipality-level analysis (N ~= 5,570) -----------------------------
# We now load the municipality-level `analysis_compliance_municipality` dataset
# instead of the state-level `analysis_compliance` (N=27). The municipality
# grain gives proper statistical power for correlation, OLS and ML work below.
#
# For backward compatibility with the rest of the notebook we:
#   1. Rename muni-level columns to the old state-level names (e.g.
#      `population_2022` -> `population`) so downstream cells work unchanged.
#   2. Regenerate the human-readable region dummies (is_norte, is_nordeste,
#      is_sudeste, is_sul, is_centro_oeste) with the same names they had in
#      the state-level dataset.
#   3. Add state dummies (is_state_<IBGE-2digit-code>) as extra features.
df = loader.load_dataset('analysis_compliance_municipality')
df = df.rename(columns={
    'population_2022': 'population',
    'literacy_rate_2022': 'avg_literacy_rate',
    'avg_income_2022': 'avg_income',
})

REGION_NAME_TO_DUMMY = {
    'Norte': 'is_norte',
    'Nordeste': 'is_nordeste',
    'Sudeste': 'is_sudeste',
    'Sul': 'is_sul',
    'Centro-Oeste': 'is_centro_oeste',
}
for _rname, _col in REGION_NAME_TO_DUMMY.items():
    df[_col] = (df['region_name'] == _rname).astype('Int64')
REGION_DUMMY_COLS = list(REGION_NAME_TO_DUMMY.values())

state_dummies = pd.get_dummies(df['state_code'], prefix='is_state').astype('Int64')
df = pd.concat([df, state_dummies], axis=1)
STATE_DUMMY_COLS = list(state_dummies.columns)

print(f"Loaded {len(df):,} observations (municipalities across {df['state_code'].nunique()} states)")
print(f"Region dummies: {REGION_DUMMY_COLS}")
print(f"State dummies: {len(STATE_DUMMY_COLS)} columns (first: {STATE_DUMMY_COLS[0]}, last: {STATE_DUMMY_COLS[-1]})")
df.head()

In [ ]:
# Feature columns at MUNICIPALITY level.
# - dropped `n_municipalities` (constant=1 at this grain)
# - dropped `is_sudeste` as the implicit base region (dummy trap)
# - added `log_total_transfers` if present -- central to the thesis question
feature_cols = ['log_income', 'avg_literacy_rate', 'log_population',
                'is_norte', 'is_nordeste', 'is_sul', 'is_centro_oeste']
if 'log_total_transfers' in df.columns:
    feature_cols.append('log_total_transfers')

# Drop rows with NaN in features or target (muni-level has some nullable cols).
_mask = df[feature_cols + ['sanctions_per_100k']].notna().all(axis=1)
X = df.loc[_mask, feature_cols].copy().astype(float)
y_regression = df.loc[_mask, 'sanctions_per_100k'].astype(float)

print(f"Features shape: {X.shape}  (N={len(X):,} municipalities after dropping NaN)")
print(f"Target shape: {y_regression.shape}")
print(f"\nFeatures: {list(X.columns)}")

## 2. Regression Models - Predict Sanctions Rate

### 2.1 Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_regression, test_size=0.25, random_state=42
)

print(f"Training set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)


### 2.2 Model Training and Comparison

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=0.1),
    'ElasticNet': ElasticNet(alpha=0.1, l1_ratio=0.5),
    'Decision Tree': DecisionTreeRegressor(max_depth=5, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, max_depth=3, random_state=42)
}

results = []

for name, model in models.items():
    print(f"Training {name}...")
    
    if name in ['Ridge', 'Lasso', 'ElasticNet']:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
    
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    results.append({
        'Model': name,
        'RMSE': rmse,
        'MAE': mae,
        'R²': r2
    })

results_df = pd.DataFrame(results).sort_values('R²', ascending=False)

print("\n" + "=" * 80)
print("REGRESSION MODEL COMPARISON")
print("=" * 80)
display(results_df)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].barh(results_df['Model'], results_df['R²'], color='steelblue', alpha=0.7)
axes[0].set_xlabel('R² Score')
axes[0].set_title('Model Performance: R²', fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='x')

axes[1].barh(results_df['Model'], results_df['RMSE'], color='coral', alpha=0.7)
axes[1].set_xlabel('RMSE')
axes[1].set_title('Model Performance: RMSE', fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='x')

axes[2].barh(results_df['Model'], results_df['MAE'], color='lightgreen', alpha=0.7)
axes[2].set_xlabel('MAE')
axes[2].set_title('Model Performance: MAE', fontweight='bold')
axes[2].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()


### 2.3 Best Model - Detailed Analysis

In [ ]:
best_model_name = results_df.iloc[0]['Model']
best_model = models[best_model_name]

print(f"Best Model: {best_model_name}")
print("=" * 70)

if best_model_name in ['Ridge', 'Lasso', 'ElasticNet']:
    y_pred_best = best_model.predict(X_test_scaled)
else:
    y_pred_best = best_model.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].scatter(y_test, y_pred_best, alpha=0.6, s=100)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0].set_xlabel('Actual Sanctions per 100k', fontsize=12)
axes[0].set_ylabel('Predicted Sanctions per 100k', fontsize=12)
axes[0].set_title(f'{best_model_name}: Actual vs Predicted', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

residuals = y_test - y_pred_best
axes[1].scatter(y_pred_best, residuals, alpha=0.6, s=100)
axes[1].axhline(y=0, color='r', linestyle='--', lw=2)
axes[1].set_xlabel('Predicted Sanctions per 100k', fontsize=12)
axes[1].set_ylabel('Residuals', fontsize=12)
axes[1].set_title(f'{best_model_name}: Residual Plot', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


### 2.4 Feature Importance (Tree-based Models)

In [ ]:
rf_model = models['Random Forest']

feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("Feature Importance (Random Forest)")
print("=" * 70)
display(feature_importance)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['Feature'], feature_importance['Importance'], color='teal', alpha=0.7)
plt.xlabel('Importance', fontsize=12)
plt.title('Feature Importance - Random Forest', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()


### 2.5 Cross-Validation

In [ ]:
cv_results = []
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    if name in ['Ridge', 'Lasso', 'ElasticNet']:
        X_cv = scaler.fit_transform(X)
    else:
        X_cv = X
    
    scores = cross_val_score(model, X_cv, y_regression, cv=kfold, 
                            scoring='r2', n_jobs=-1)
    
    cv_results.append({
        'Model': name,
        'Mean R²': scores.mean(),
        'Std R²': scores.std(),
        'Min R²': scores.min(),
        'Max R²': scores.max()
    })

cv_df = pd.DataFrame(cv_results).sort_values('Mean R²', ascending=False)

print("5-Fold Cross-Validation Results")
print("=" * 80)
display(cv_df)


## 3. Classification Models - High/Low Compliance Risk

### 3.1 Create Binary Target

In [ ]:
# Binary target definition (municipality-level).
#
# DATA CAVEAT: in the current Gold snapshot ~78% of municipalities have
# `n_sanctions == 0`, so the median of `sanctions_per_100k` is 0. Using
# `> median` as the boundary collapses to "has any sanction at all", which
# IS actually a meaningful binary target (did this muni ever show up in
# CEIS / CNEP / CEPIM?). We therefore define:
#     high_risk == 1  iff  the muni has at least one sanction registered.
# Class balance is ~22/78, so downstream classifiers below use
# class_weight='balanced' (and we track ROC-AUC / F1, not just accuracy).
df_sub = df.loc[_mask].copy()
df_sub['high_risk'] = (df_sub['n_sanctions'] > 0).astype(int)

print("Binary target: has at least one sanction registered")
print(f"Class distribution:")
print(df_sub['high_risk'].value_counts())
print(f"\nClass balance:")
print(df_sub['high_risk'].value_counts(normalize=True).round(3))

y_classification = df_sub['high_risk'].copy()

### 3.2 Train Classification Models

In [ ]:
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X, y_classification, test_size=0.25, random_state=42, stratify=y_classification
)

scaler_clf = StandardScaler()
X_train_clf_scaled = scaler_clf.fit_transform(X_train_clf)
X_test_clf_scaled = scaler_clf.transform(X_test_clf)

print(f"Training set: {len(X_train_clf)} samples")
print(f"Test set: {len(X_test_clf)} samples")


In [ ]:
# class_weight='balanced' compensates for the ~78/22 imbalance (only 22%
# of munis have a sanction registered). Without it, classifiers would trivially
# predict 0 everywhere and get 78% accuracy while missing every positive case.
clf_models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000,
                                              class_weight='balanced'),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=5,
                                            random_state=42, n_jobs=-1,
                                            class_weight='balanced'),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100,
                                                    max_depth=3, random_state=42),
}

clf_results = []

for name, model in clf_models.items():
    print(f"\nTraining {name}...")

    if name == 'Logistic Regression':
        model.fit(X_train_clf_scaled, y_train_clf)
        y_pred = model.predict(X_test_clf_scaled)
        y_pred_proba = model.predict_proba(X_test_clf_scaled)[:, 1]
    else:
        model.fit(X_train_clf, y_train_clf)
        y_pred = model.predict(X_test_clf)
        y_pred_proba = model.predict_proba(X_test_clf)[:, 1]

    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

    accuracy = accuracy_score(y_test_clf, y_pred)
    precision = precision_score(y_test_clf, y_pred, zero_division=0)
    recall = recall_score(y_test_clf, y_pred, zero_division=0)
    f1 = f1_score(y_test_clf, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_test_clf, y_pred_proba)

    clf_results.append({
        'Model': name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'ROC-AUC': roc_auc,
    })

    print(f"\n{name} - Classification Report:")
    print(classification_report(y_test_clf, y_pred,
                                target_names=['No sanctions', 'Has sanctions'],
                                zero_division=0))

clf_results_df = pd.DataFrame(clf_results).sort_values('ROC-AUC', ascending=False)

print("\n" + "=" * 80)
print("CLASSIFICATION MODEL COMPARISON (imbalanced, class_weight='balanced')")
print("=" * 80)
display(clf_results_df)

### 3.3 Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (name, model) in enumerate(clf_models.items()):
    if name == 'Logistic Regression':
        y_pred = model.predict(X_test_clf_scaled)
    else:
        y_pred = model.predict(X_test_clf)
    
    cm = confusion_matrix(y_test_clf, y_pred)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=['Low Risk', 'High Risk'],
                yticklabels=['Low Risk', 'High Risk'])
    axes[idx].set_title(f'{name}\nConfusion Matrix', fontweight='bold')
    axes[idx].set_ylabel('Actual')
    axes[idx].set_xlabel('Predicted')

plt.tight_layout()
plt.show()


### 3.4 ROC Curves

In [ ]:
plt.figure(figsize=(10, 8))

for name, model in clf_models.items():
    if name == 'Logistic Regression':
        y_pred_proba = model.predict_proba(X_test_clf_scaled)[:, 1]
    else:
        y_pred_proba = model.predict_proba(X_test_clf)[:, 1]
    
    fpr, tpr, _ = roc_curve(y_test_clf, y_pred_proba)
    auc = roc_auc_score(y_test_clf, y_pred_proba)
    
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})', linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier', linewidth=2)
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves - Compliance Risk Classification', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 4. Model Interpretation

### 4.1 Logistic Regression Coefficients

In [ ]:
lr_model = clf_models['Logistic Regression']

coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': lr_model.coef_[0],
    'Odds Ratio': np.exp(lr_model.coef_[0])
}).sort_values('Coefficient', key=abs, ascending=False)

print("Logistic Regression Coefficients")
print("=" * 70)
display(coef_df)

plt.figure(figsize=(10, 6))
colors = ['red' if c < 0 else 'green' for c in coef_df['Coefficient']]
plt.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors, alpha=0.7)
plt.axvline(x=0, color='black', linestyle='--', linewidth=1)
plt.xlabel('Coefficient Value', fontsize=12)
plt.title('Logistic Regression Coefficients\n(Red = Negative, Green = Positive)', 
          fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()


### 4.2 Feature Importance (Classification)

In [ ]:
rf_clf = clf_models['Random Forest']

feature_importance_clf = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_clf.feature_importances_
}).sort_values('Importance', ascending=False)

print("Feature Importance (Random Forest Classifier)")
print("=" * 70)
display(feature_importance_clf)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance_clf['Feature'], feature_importance_clf['Importance'], 
         color='purple', alpha=0.7)
plt.xlabel('Importance', fontsize=12)
plt.title('Feature Importance - Random Forest Classifier', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()


## 5. Summary and Recommendations

### 5.1 Regression Models
- **Best model**: ElasticNet, selected via cross-validation among Linear Regression, Ridge, Lasso, ElasticNet, Decision Tree, Random Forest, and Gradient Boosting.
- **Key predictors**: Log income and regional dummies (Norte, Nordeste) are the most important features for predicting sanctions per 100k.
- **Random Forest feature importance** confirms income as the dominant predictor, followed by population and literacy indicators.
- **Limitation**: With only 27 observations (states), all models face high variance and limited generalizability.

### 5.2 Classification Models
- **Binary target**: States classified as high-risk (above median sanctions per 100k of 12.22) vs low-risk.
- **Class balance**: Nearly balanced (14 low-risk, 13 high-risk states).
- **Logistic Regression** achieved the best F1 performance (weighted F1 = 0.51), but with poor recall on Low Risk class (0.25).
- **Random Forest and Gradient Boosting** performed worse (accuracy 0.43 and 0.29 respectively), likely due to overfitting on the small dataset.
- **Overall**: Classification performance is weak across all models due to the small sample size (n = 27). Results should be treated as exploratory.

### 5.3 Key Insights
- Income is consistently the strongest predictor of sanctions rates across all modeling approaches.
- Regional effects (Norte, Nordeste) persist after controlling for socioeconomic indicators, suggesting institutional or governance factors at play.
- The Distrito Federal remains a strong outlier that influences all models.

### 5.4 Limitations and Caveats
- **Small sample** (n = 27): Insufficient for robust ML model training. Cross-validation helps but cannot fully compensate.
- **Correlation, not causation**: Higher income may reflect greater institutional capacity to detect violations, not more actual misconduct.
- **State-level aggregation** masks within-state variation — municipality-level analysis (NB04) provides finer granularity.
- **No temporal dimension**: Models are cross-sectional snapshots, not predictive of future sanctions trends.
